In [6]:
import torch
import torch.nn as nn
import math

In [75]:
class PositionaEncoding(nn.Module):
    def __init__(self, vocab_size, d_rpr, *args, **kwargs):
        super().__init__(*args, **kwargs)
        pe = torch.zeros(vocab_size, d_rpr)
        position = torch.arange(vocab_size).unsqueeze(1)
        denom = torch.exp(torch.arange(0, d_rpr, 2) * (-math.log(10000.0) / d_rpr))
        # print(position,denom, position * denom)
        pe[:,::2] = torch.sin(position * denom)
        pe[:,1::2] = torch.cos(position * denom)

        pe = pe.unsqueeze(0)
        self.register_buffer('pe', pe)
    def forward(self,x):
        x += self.pe[:, :x.size(1)]
        return x

a = PositionaEncoding(4,6)
a.forward(torch.rand((1,1,6)))


tensor([[[0.4955, 1.1141, 0.4634, 1.9808, 0.2203, 1.8484]]])

In [79]:
class TransformerModel(nn.Module):
    def __init__(
        self,
        src_vocab_size,
        tgt_vocab_size,
        d_rpr=512,
        nhead=8,
        num_layers=6,
        d_ff=2048,
        dropout=0.1,
    ):
        super().__init__()
        self.d_rpr = d_rpr

        self.source_embedding = nn.Embedding(src_vocab_size, d_rpr)
        self.target_embedding = nn.Embedding(tgt_vocab_size, d_rpr)
        self.pos_encoder = PositionaEncoding(5000, d_rpr)

        self.transformer = nn.Transformer(
            d_model=d_rpr,
            nhead=nhead,
            num_encoder_layers=num_layers,
            num_decoder_layers=num_layers,
            dim_feedforward=d_ff,
            dropout=dropout,
            batch_first=True,
        )
        self.fc_out = nn.Linear(d_rpr, tgt_vocab_size)

    def forward(self, src, tgt, src_mask=None, tgt_mask=None, memory_mask=None):
        src = self.source_embedding(src) * math.sqrt(self.d_rpr)
        tgt = self.target_embedding(tgt) * math.sqrt(self.d_rpr)

        src = self.pos_encoder(src)
        tgt = self.pos_encoder(tgt)

        output = self.transformer(
            src, tgt, src_mask=src_mask, tgt_mask=tgt_mask, memory_mask=memory_mask
        )
        return self.fc_out(output)

a = TransformerModel(src_vocab_size=10, tgt_vocab_size=5)
a.pos_encoder.pe

tensor([[[ 0.0000e+00,  1.0000e+00,  0.0000e+00,  ...,  1.0000e+00,
           0.0000e+00,  1.0000e+00],
         [ 8.4147e-01,  5.4030e-01,  8.2186e-01,  ...,  1.0000e+00,
           1.0366e-04,  1.0000e+00],
         [ 9.0930e-01, -4.1615e-01,  9.3641e-01,  ...,  1.0000e+00,
           2.0733e-04,  1.0000e+00],
         ...,
         [ 9.5625e-01, -2.9254e-01,  9.3594e-01,  ...,  8.5926e-01,
           4.9515e-01,  8.6881e-01],
         [ 2.7050e-01, -9.6272e-01,  8.2251e-01,  ...,  8.5920e-01,
           4.9524e-01,  8.6876e-01],
         [-6.6395e-01, -7.4778e-01,  1.4615e-03,  ...,  8.5915e-01,
           4.9533e-01,  8.6871e-01]]])

In [80]:
src = torch.randint(0, 100, (32, 10))  # (batch_size, src_seq_len)
tgt = torch.randint(0, 100, (32, 9))   # (batch_size, tgt_seq_len)

model = TransformerModel(src_vocab_size=100, tgt_vocab_size=100)
out = model(src, tgt)
print(out.shape)  # Expected: (batch_size, tgt_seq_len, tgt_vocab_size)

torch.Size([32, 9, 100])


In [2]:
import streamlit
from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain_ollama.llms import OllamaLLM
from langchain.vectorstores import FAISS
from langchain.text_splitter import RecursiveCharacterTextSplitter
import whisper
import os
import shutil

template = """
You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, just say that you don't know. Use three sentences maximum and keep the answer concise.
Question: {question} 
Context: {context} 
Answer:
"""
audio_directory = "audio"
embeding = OllamaEmbeddings(model="all-minilm:l6-v2", base_url="http://localhost:11434")
model = OllamaLLM(model="Gemma3:4b")


def upload_audio(file):
    os.makedirs(audio_directory, exist_ok=True)
    try: # if uploaded via streamlit UI
        with open(audio_directory + file.name, "wb") as f:
            f.write(file.getbuffer())
            return audio_directory + file.name
    except:
        shutil.copy(file, os.path.join(audio_directory, os.path.basename(file)))
        return os.path.join(audio_directory, os.path.basename(file))

def transcript_audio(file_path):
    whisper_model = whisper.load_model("medium.en")
    trans = whisper_model.transcribe(file_path)
    return trans["text"]


def split_text(text):
    txt_splitter = RecursiveCharacterTextSplitter(
        chunk_size=1000,
        chunk_overlap = 200,
        add_start_index=True
    )

    return txt_splitter.split_text(text)


def store_vector(texts):
    vs = FAISS.from_texts(texts, )
    return vs.as_retriever()



In [4]:


file_to_upload = r"C:\Users\mamma\OneDrive\Pictures\abrahamlincolnmystic_2_grierson_64kb.mp3"
file = upload_audio(file_to_upload)

In [5]:
file = 'audio\\abrahamlincolnmystic_2_grierson_64kb.mp3'

In [6]:
a = transcript_audio(file)

In [7]:
texts = split_text(a)

In [9]:
vector_db = FAISS.from_texts(texts, embeding)
retrieved = vector_db.as_retriever(search_type="similarity", search_kwargs={"k": 4})

In [8]:
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"


In [23]:
relevant_documents = retrieved.invoke("what is the bset character document about? ")
relevant_documents


[Document(id='7ce48677-1220-4b8a-b752-eb93792e74b2', metadata={}, page_content='which confers upon his greatness in his own person, in a real form, that is, in a finite, positive, visible, and determinate form, so that what is general may not suppress what is particular, and which is particular may not dissipate and dissolve what is general, that the infinite and the finite may be blended together in that proportion which truly constitutes human greatness. All of which applies to Lincoln. "\'Conceive a great machine,\' wrote Geezo, the historian, the design of which is centered in a single mind, though its various parts are entrusted to various workmen, separated from and strangers to each other. No one of them understands the work as a whole, nor the general result which he concerts in producing, but every one executes with intelligence and freedom by rational and voluntary acts the particular task assigned to him. It is thus by the hand of man the designs of Providence are wrought ou

In [21]:
relevant_documents

[Document(id='7ce48677-1220-4b8a-b752-eb93792e74b2', metadata={}, page_content='which confers upon his greatness in his own person, in a real form, that is, in a finite, positive, visible, and determinate form, so that what is general may not suppress what is particular, and which is particular may not dissipate and dissolve what is general, that the infinite and the finite may be blended together in that proportion which truly constitutes human greatness. All of which applies to Lincoln. "\'Conceive a great machine,\' wrote Geezo, the historian, the design of which is centered in a single mind, though its various parts are entrusted to various workmen, separated from and strangers to each other. No one of them understands the work as a whole, nor the general result which he concerts in producing, but every one executes with intelligence and freedom by rational and voluntary acts the particular task assigned to him. It is thus by the hand of man the designs of Providence are wrought ou

In [2]:
import torch
print(torch.__version__)
print(torch.version.cuda)
print(torch.backends.cudnn.enabled)


2.7.1+cu126
12.6
True


In [3]:
torch.cuda.is_available()


True